# Local LoCoMo memory-architecture evaluation

Run this notebook in a GPU-enabled Google Colab runtime. It executes the local 4-bit evaluator in this repository; no inference API is used. Keep the experimental parameters identical for all models within a comparison.

In [ ]:
# Run once per fresh Colab runtime. Colab already supplies CUDA PyTorch.
from pathlib import Path
import os, subprocess, sys

PROJECT_DIR = Path.cwd().resolve()
if not (PROJECT_DIR / 'locomo10.json').exists():
    default_repo = Path('/content/The_Polymath_Initiative')
    if default_repo.exists():
        PROJECT_DIR = default_repo
    else:
        raise FileNotFoundError('Open this notebook from the repository root, or set PROJECT_DIR to the clone containing locomo10.json.')

os.chdir(PROJECT_DIR)
print('Project:', PROJECT_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'colab_code/requirements-colab.txt'], check=True)

In [ ]:
# Verify that Colab assigned a GPU before loading any model.
import torch
assert torch.cuda.is_available(), 'No GPU detected. In Colab: Runtime > Change runtime type > T4 GPU (or better), then reconnect.'
print('PyTorch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM (GiB):', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## Optional Hugging Face authentication

Qwen models need no token. Run the next cell only for the gated Llama models, after accepting each model license on Hugging Face.

In [ ]:
# Optional: run only for llama3_2_3b or llama3_1_8b.
from getpass import getpass
if not os.environ.get('HF_TOKEN'):
    os.environ['HF_TOKEN'] = getpass('Hugging Face token (input hidden): ')
print('HF_TOKEN is available to this notebook process.')

## Smoke test

Run this first. It evaluates one LoCoMo conversation with the smallest primary model, verifies model download and 4-bit loading, then writes a checkpointed result file.

In [ ]:
smoke_command = [
    sys.executable, 'colab_code/run_local_evaluation.py',
    '--model-key', 'qwen2_5_1_5b',
    '--experiment', 'raw_truncated',
    '--out-file', 'results/smoke.json',
    '--context-token-budget', '6000',
    '--max-samples', '1',
]
subprocess.run(smoke_command, check=True)

## Run one experimental cell

Edit only `MODEL_KEY` and `EXPERIMENT` below. Do **not** change the 6,000-token window, top-k, or tier working-set size between model sizes when comparing a row. The output contains the full run configuration and is checkpointed after every completed conversation.

In [ ]:
MODEL_KEY = 'qwen2_5_7b'  # qwen2_5_1_5b, qwen2_5_7b, qwen2_5_14b, llama3_2_3b, llama3_1_8b
EXPERIMENT = 'summary_rag' # raw_truncated, raw_full, summary_rag, facts_rag, summary_tiered, graph_traversal
CONTEXT_TOKEN_BUDGET = 6000
TOP_K = 5
TIER_WORKING_TURNS = 12

assert EXPERIMENT != 'raw_full' or MODEL_KEY, 'raw_full must only be run when the complete prompt fits GPU memory; never silently truncate it.'
command = [
    sys.executable, 'colab_code/run_local_evaluation.py',
    '--model-key', MODEL_KEY,
    '--experiment', EXPERIMENT,
    '--out-file', f'results/{MODEL_KEY}__{EXPERIMENT}.json',
    '--emb-dir', f'memory_cache/{MODEL_KEY}',
    '--context-token-budget', str(CONTEXT_TOKEN_BUDGET),
    '--top-k', str(TOP_K),
    '--tier-working-turns', str(TIER_WORKING_TURNS),
    '--log-memory-trace',
]
print(' '.join(command))
subprocess.run(command, check=True)

## Primary Qwen matrix

This cell runs the four core conditions for Qwen 1.5B, 7B, and 14B. It launches a new Python process per cell, which releases model memory before the next model loads. `raw_full` and graph traversal are deliberately excluded here: enable and run them separately only when appropriate for the runtime and your preregistered scope.

In [ ]:
PRIMARY_MODELS = ['qwen2_5_1_5b', 'qwen2_5_7b', 'qwen2_5_14b']
CORE_EXPERIMENTS = ['raw_truncated', 'summary_rag', 'facts_rag', 'summary_tiered']

for model_key in PRIMARY_MODELS:
    for experiment in CORE_EXPERIMENTS:
        command = [
            sys.executable, 'colab_code/run_local_evaluation.py',
            '--model-key', model_key,
            '--experiment', experiment,
            '--out-file', f'results/{model_key}__{experiment}.json',
            '--emb-dir', f'memory_cache/{model_key}',
            '--context-token-budget', '6000',
            '--top-k', '5',
            '--tier-working-turns', '12',
            '--log-memory-trace',
        ]
        print(f'\n=== {model_key} | {experiment} ===')
        subprocess.run(command, check=True)

In [ ]:
# Inspect completed result and stats files.
for result in sorted((PROJECT_DIR / 'results').glob('*.json')):
    print(result.relative_to(PROJECT_DIR), f'{result.stat().st_size / 1024:.1f} KiB')